# 05 · Porous media with Brinkmanlets

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fairyflies9/fluidlets/blob/main/tutorials/05_porous_media_brinkmanlets.ipynb)

Many biological fluids are threaded with a sparse mesh of fibres: the glycocalyx on blood vessel
walls, mucus, biofilms, the extracellular matrix. Averaged over the fibres, the mesh pulls back on
the fluid with a force proportional to the local velocity. Adding that drag to the Stokes equations
gives the **Brinkman equations**

$$
\mu\Delta\mathbf u - \mu\alpha^2\mathbf u - \nabla p = -\mathbf f\,\phi_\varepsilon,
\qquad \nabla\cdot\mathbf u = 0 .
$$

Here $\alpha^2 = \mu/k$ with $k$ the permeability, and $1/\alpha$ is the **screening length**,
roughly the spacing of the fibres. With $\alpha = 0$ there is no mesh and we are back to Stokes flow.

This notebook follows Cortez, Cummins, Leiderman & Varela, *Computation of three-dimensional Brinkman
flows using regularized methods*, J. Comput. Phys. **229**, 7609 (2010), and uses
`fluidlets.Brinkmanlet3D`, their "regularization 1".

**You will learn**

* how the Brinkmanlet is built from one scalar function $B(r)$, and how it reduces to the Stokeslet,
* why the formula has to be evaluated carefully when $\alpha r$ is small,
* what *screening* does to the flow of a point force: faster decay and backflow,
* how to compute the drag on a sphere in a porous medium and check it against the exact answer,
* why $\varepsilon$ and the point spacing $h$ must resolve the screening length $1/\alpha$.

In [ ]:
# On Colab or Binder, install Fluidlets first (skipped when it is already installed).
import importlib.util, subprocess, sys
if importlib.util.find_spec("fluidlets") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/fairyflies9/fluidlets"])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import fluidlets as fl
from fluidlets.geometry import fibonacci_sphere

mu = 1.0
ez = np.array([0.0, 0.0, 1.0])

## 1. One scalar function

For a radial blob, both the Stokeslet and the Brinkmanlet can be written as

$$
\mathbf u(\mathbf x) = H_1(r)\,\mathbf f + H_2(r)\,(\mathbf f\cdot\mathbf x)\,\mathbf x,
\qquad
H_1 = -\frac{rB'' + B'}{r},\quad H_2 = \frac{rB'' - B'}{r^3},
$$

where $B$ solves $(\Delta - \alpha^2)B = G$ and $\Delta G = \phi$ (CCLV 2010, Eqs. 1–4). For a point
force (the singular case),

$$
B(r) = \frac{1 - e^{-\alpha r}}{4\pi\alpha^2 r},
$$

and working out the derivatives with $z = \alpha r$ gives

$$
h_1(r) = \frac{e^{-z}\left(1 + \frac1z + \frac1{z^2}\right) - \frac1{z^2}}{4\pi r},
\qquad
h_2(r) = \frac{\frac{3}{z^2} - e^{-z}\left(1 + \frac3z + \frac3{z^2}\right)}{4\pi r^3}.
$$

**Regularization.** CCLV's regularization 1 replaces $r$ by $R = \sqrt{r^2 + \varepsilon^2}$ inside
$B$. The chain rule then gives

$$
H_2(r) = h_2(R), \qquad H_1(r) = h_1(R) + \varepsilon^2 h_2(R).
$$

As $\alpha\to0$, both brackets tend to $1/2$, so $h_1\to 1/(8\pi R)$ and $h_2\to1/(8\pi R^3)$. Then
$H_1 = (r^2 + 2\varepsilon^2)/(8\pi R^3)$: exactly the regularized Stokeslet of tutorial 01. Let's check.

In [ ]:
rng = np.random.default_rng(0)
targets, sources, forces = rng.normal(size=(50, 3)), rng.normal(size=(20, 3)), rng.normal(size=(20, 3))

u_stokes = fl.evaluate(fl.Stokeslet3D(), targets, sources, forces, 0.1, mu)
u_brink0 = fl.evaluate(fl.Brinkmanlet3D(0.0), targets, sources, forces, 0.1, mu)
print("max difference, Brinkmanlet3D(0) vs Stokeslet3D:", np.abs(u_brink0 - u_stokes).max())

## 2. A numerical trap: cancellation at small $\alpha r$

Look at the bracket in $h_1$ when $z = \alpha r$ is small: $e^{-z}/z^2$ and $1/z^2$ are both huge and
nearly equal, and we subtract them to get something close to $1/2$. In floating point that throws away
digits. Fluidlets switches to the Taylor series

$$
e^{-z}\left(1 + \tfrac1z + \tfrac1{z^2}\right) - \tfrac1{z^2}
= \sum_{n\ge0} (-1)^n\left[\frac{1}{(n+2)!} - \frac{1}{(n+1)!} + \frac1{n!}\right] z^n
= \tfrac12 - \tfrac23 z + \tfrac38 z^2 - \dots
$$

for $z < 0.5$. Compare the two:

In [ ]:
from math import factorial
from fluidlets.kernels.brinkmanlet3d import f1      # the bracket in h1, as Fluidlets computes it

def naive(z):
    return np.exp(-z) * (1 + 1 / z + 1 / z**2) - 1 / z**2

def series(z, terms=6):                             # a few Taylor terms are plenty for tiny z
    return sum((-1) ** n * (1 / factorial(n + 2) - 1 / factorial(n + 1) + 1 / factorial(n)) * z**n
               for n in range(terms))

print("      z        naive closed form      Fluidlets           series (6 terms)")
for z in [1.0, 1e-1, 1e-3, 1e-5, 1e-7]:
    print(f"{z:8.0e}   {naive(z):.16f}   {f1(z):.16f}   {series(z):.16f}")

At $z = 10^{-5}$ the naive formula is already wrong in the sixth digit, and at $10^{-7}$ it has
lost every digit beyond $1/2$. The error grows like $10^{-16}/z^2$. (Going the other way, the
six-term series is visibly off at $z = 1$, which is why Fluidlets uses the closed form for
$z\ge0.5$ and a longer series below.) This matters in practice: small
$\alpha$ (a very open mesh) or a target close to a source both make $z$ small.

## 3. Screening: what the mesh does to the flow

Push on the fluid at the origin with $\mathbf f = \mathbf e_z$ and look at $u_z$ along the $x$-axis,
perpendicular to the force. In Stokes flow $u_z = 1/(8\pi\mu r)$, a slow $1/r$ decay. We plot
$8\pi\mu r\,u_z$, which is 1 for the Stokeslet.

In [ ]:
r = np.logspace(-1.5, 1.3, 300)
line = np.c_[r, np.zeros_like(r), np.zeros_like(r)]

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for alpha, c in zip([0.0, 0.5, 2.0, 8.0], ["k", "C0", "C1", "C3"]):
    uz = fl.evaluate(fl.Brinkmanlet3D(alpha), line, [[0, 0, 0]], [ez], 0.005, mu)[:, 2]
    axs[0].semilogx(r, 8 * np.pi * mu * r * uz, color=c, label=rf"$\alpha$ = {alpha:g}")
    axs[1].loglog(r, np.abs(uz), color=c, label=rf"$\alpha$ = {alpha:g}")
    if alpha > 0:
        axs[0].axvline(1 / alpha, color=c, ls=":", lw=0.8)
axs[0].axhline(0, color="gray", lw=0.5)
axs[0].set_xlabel("distance r"); axs[0].set_ylabel(r"$8\pi\mu\, r\, u_z$")
axs[0].set_title(r"dotted lines: screening length $1/\alpha$"); axs[0].legend()
axs[1].loglog(r, 1 / (8 * np.pi * r), "k--", lw=0.8, label=r"$1/(8\pi r)$")
axs[1].loglog(r, 1 / (4 * np.pi * 8.0**2 * r**3), "C3--", lw=0.8, label=r"$1/(4\pi\alpha^2 r^3)$, $\alpha$ = 8")
axs[1].set_xlabel("distance r"); axs[1].set_ylabel(r"$|u_z|$"); axs[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

Two things happen beyond the screening length $1/\alpha$:

* **Faster decay.** Far away the exponentials die out and only the algebraic terms are left:
  $h_1\to -1/(4\pi\alpha^2 r^3)$ and $h_2\to 3/(4\pi\alpha^2 r^5)$. The mesh absorbs the momentum the
  force puts in, so the flow falls off like $1/r^3$ (right panel), like the flow of a dipole, instead
  of the Stokeslet's $1/r$.
* **Backflow.** $h_1 < 0$ far away, so to the side of the force the fluid moves *against* it (the
  curves in the left panel cross zero; the dips in the right panel are those sign changes). The fluid
  pushed forward near the force returns around the sides in a short loop, instead of in the huge
  circulation of Stokes flow.

A Brinkmanlet therefore only "talks" to its neighbours within a few screening lengths, and that will
matter for the discretization below.

## 4. A sphere in a porous medium

Hold a sphere of radius $a$ still in a uniform stream $\mathbf U$ through the mesh (equivalently,
for the disturbance flow, prescribe velocity $\mathbf U$ on its surface) and solve the resistance
problem. CCLV 2010 (Sec. 3.1) compare the summed regularized forces with

$$
F = 6\pi\mu a U\left(1 + \alpha a + \frac{\alpha^2 a^2}{3}\right).
$$

With $\alpha = 0$ this is Stokes' law. The $\alpha a$ and $\alpha^2a^2$ terms are the extra resistance
of the mesh. (The drag on a sphere *translating* through the medium has $\alpha^2a^2/9$ instead. The
difference, $\alpha^2\mu\,\tfrac43\pi a^3 U$, is the Darcy resistance of the fluid that would fill the
sphere's volume, which the regularized forces also hold back.)

We use 1500 points from `fibonacci_sphere`, point spacing $h = \sqrt{4\pi a^2/N}$, and
$\varepsilon = 0.4h$, about the choice tutorial 02 recommends for Stokes flow.

In [ ]:
a, N = 1.0, 1500
X = fibonacci_sphere(N, a)
h = np.sqrt(4 * np.pi * a**2 / N)

def exact_force(alpha):
    return 6 * np.pi * mu * a * (1 + alpha * a + (alpha * a) ** 2 / 3)

def sphere_force(alpha, eps):
    forces, _ = fl.resistance(fl.Brinkmanlet3D(alpha), X, np.tile(ez, (N, 1)), eps, mu, method="lu")
    return forces[:, 2].sum()

print(f"N = {N}, h = {h:.3f}, eps = 0.4 h = {0.4 * h:.4f}\n")
print("  alpha  1/alpha   1/(alpha h)   F computed   F exact    error")
for alpha in [0.0, 0.5, 1.0, 2.0, 4.0, 10.0]:
    F = sphere_force(alpha, 0.4 * h)
    inv = np.inf if alpha == 0 else 1 / alpha
    print(f"{alpha:6.1f}  {inv:7.3f}  {inv / h:10.1f}  {F:11.3f} {exact_force(alpha):10.3f}  {F / exact_force(alpha) - 1:+7.2%}")

For an open mesh ($\alpha\lesssim1$) the answer is as good as in Stokes flow. But as the screening
length $1/\alpha$ shrinks toward the point spacing, the computed force falls short: by about 9 % at
$\alpha = 4$, where $1/\alpha$ is less than three point spacings, and by almost 40 % at $\alpha = 10$.

The reason is section 3. Each regularized Brinkmanlet only influences the fluid within a few
screening lengths. When that is comparable to the gaps between points, the forces can no longer
hold back the fluid *between* the points: the discretized sphere leaks, and a leaky sphere feels
less drag. In Stokes flow the long $1/r$ reach of every Stokeslet hides this; in a porous medium it
does not.

## 5. Choosing ε when there is a screening length

A larger blob covers the gaps. Let us find the $\varepsilon$ that makes the force exact, for a few
values of $\alpha$, by bisection (the computed force increases with $\varepsilon$).

In [ ]:
def eps_for_exact_force(alpha, lo=0.2, hi=1.6, iters=9):
    """eps (as a multiple of h) at which the computed force equals the exact one."""
    target = exact_force(alpha)
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if sphere_force(alpha, mid * h) < target else (lo, mid)
    return 0.5 * (lo + hi)

for alpha in [0.5, 2.0, 4.0]:
    print(f"alpha = {alpha:3.1f}   alpha h = {alpha * h:.3f}   eps* = {eps_for_exact_force(alpha):.2f} h")

The best blob grows as $\alpha h$ grows. CCLV saw the same thing with $N = 7996$: their
optimal $\varepsilon$ for the drag rose from 0.016 at $\alpha = 0.1$ to 0.045 at $\alpha = 20$
(`examples/sphere_in_brinkman` reproduces their table).

> **Rule of thumb.** Resolve the screening length: keep the point spacing $h$ well below $1/\alpha$
> (several points per screening length), and check $\varepsilon$ on a problem with a known answer
> at the $\alpha$ you care about. A fixed $\varepsilon/h$ that works for Stokes flow is not enough.

Tuning $\varepsilon$ to one quantity is not free accuracy, though. CCLV found that the
$\varepsilon$ that gets the torque on a rotating sphere right is several times *smaller* than the
one for the drag. Once the discretization is coarse, different quantities prefer different blobs.

## 6. The flow around a spinning sphere

A sphere spinning at $\boldsymbol\Omega$ in a Brinkman medium drives a purely rotational flow,
$\mathbf u = (\boldsymbol\Omega\times\mathbf x)\, g(r)$, with (CCLV 2010, Sec. 3.2)

$$
g(r) = \frac{e^{\alpha(a - r)}\,a^3(1 + \alpha r)}{r^3(1 + \alpha a)} .
$$

For $\alpha = 0$ this is the Stokes result $a^3/r^3$. Let's compare it with the computed flow.

In [ ]:
r = np.linspace(1.05, 3.0, 60) * a
line = np.c_[r, np.zeros_like(r), np.zeros_like(r)]

fig, ax = plt.subplots(figsize=(6.5, 4))
for alpha, c in zip([0.0, 1.0, 4.0], ["k", "C0", "C3"]):
    K = fl.Brinkmanlet3D(alpha)
    forces, _ = fl.resistance(K, X, np.cross(ez, X), 0.4 * h, mu, method="lu")
    g_num = fl.evaluate(K, line, X, forces, 0.4 * h, mu)[:, 1] / r
    g_exact = np.exp(alpha * (a - r)) * a**3 * (1 + alpha * r) / (r**3 * (1 + alpha * a))
    ax.plot(r, g_exact, color=c, label=rf"exact, $\alpha$ = {alpha:g}")
    ax.plot(r[::4], g_num[::4], "o", color=c, mfc="none")
    print(f"alpha = {alpha:3.1f}: max |g_computed - g_exact| = {np.abs(g_num - g_exact).max():.3f}")
ax.set_xlabel("r / a"); ax.set_ylabel("angular velocity of the fluid, g(r)")
ax.set_title("circles: Fluidlets (N = 1500)"); ax.legend(); plt.show()

Away from the surface the computed flow follows the exact curves, and you can see the screening
directly: half a radius from the surface, the fluid still spins at about a third of the sphere's
rate in Stokes flow, but at only about a twentieth with $\alpha = 4$.

## Summary

* The regularized Brinkmanlet comes from a single function $B(r)$; with regularization 1 it becomes
  $H_2 = h_2(R)$, $H_1 = h_1(R) + \varepsilon^2h_2(R)$, and with $\alpha = 0$ it is exactly the
  regularized Stokeslet.
* Its formula cancels catastrophically for small $\alpha r$; a Taylor series fixes that.
* A porous medium screens the flow: beyond $1/\alpha$ it decays like $1/r^3$, with backflow to the
  side of a force.
* Because of screening, the point spacing must resolve $1/\alpha$, and the best $\varepsilon$ grows
  with $\alpha h$.

## Exercises

1. **Far field.** Show from the formulas for $h_1$ and $h_2$ that, for $\alpha r\gg1$,
   $\mathbf u\to\frac{1}{4\pi\mu\alpha^2}\left(-\frac{\mathbf f}{r^3} + \frac{3(\mathbf f\cdot\mathbf x)\mathbf x}{r^5}\right)$.
   Check it numerically with `fl.evaluate` at $r = 20$, $\alpha = 2$.
2. **Where is the backflow?** For a point force $\mathbf f = \mathbf e_z$ and $\alpha = 1$, find the
   distance along the $x$-axis at which $u_z$ changes sign. Show that it scales like $1/\alpha$.
3. **Resolution.** Repeat the table of section 4 with $N = 4000$ points. At which $\alpha$ does the
   error at $\varepsilon = 0.4h$ first exceed 5 %? Compare $1/(\alpha h)$ at that point with the
   $N = 1500$ case.
4. **Two spheres.** Place two unit spheres 3 radii apart along $x$ and move both with
   $\mathbf U = \mathbf e_z$. In Stokes flow each feels less drag than it would alone, because each
   one drags fluid along for the other. How does this hydrodynamic interaction change at
   $\alpha = 2$? Explain with section 3.
5. **Unsteady Stokes.** Oscillating flow $\mathbf u\,e^{-i\omega t}$ obeys the Brinkman equation with
   $\alpha^2 = -i\omega\rho/\mu$, a *complex* $\alpha$. `Brinkmanlet3D` accepts only real
   $\alpha\ge0$ for now. Which parts of the formulas in section 1 would you have to change, and which
   still work? (Hint: think about $e^{-z}$, the series, and which square root of $\alpha^2$ to take.)